<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); padding: 45px 20px; border-radius: 12px; text-align: center; font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; box-shadow: 0 12px 24px rgba(0,0,0,0.6); border: 1px solid rgba(255,255,255,0.05);">
    <h1 style="color: #ffffff; font-size: 40px; margin: 0 0 12px 0; font-weight: 800; letter-spacing: 1.5px; text-shadow: 0px 4px 10px rgba(0,0,0,0.8);">March Machine Learning Mania 2026</h1>
    <div style="height: 4px; width: 80px; background: linear-gradient(to right, #00c6ff, #0072ff); margin: 0 auto 15px auto; border-radius: 2px; box-shadow: 0 0 10px rgba(0, 198, 255, 0.5);"></div>
    <p style="color: #b0c4de; font-size: 16px; margin: 0; letter-spacing: 3px; text-transform: uppercase; font-weight: 500;">NCAA Tournament Prediction Models</p>
</div>

In [1]:
"""
================================================================
MARCH MACHINE LEARNING MANIA 2026
Professional Prediction Pipeline  ·  v1.0
================================================================
Author   : Imaad Mahmood
Target   : Top Leaderboard (minimise Brier Score)
Metric   : Brier Score  (lower = better; 0.25 = random)

Feature Stack
─────────────
  • Elo Ratings       — margin-adjusted K, home-court correction,
                        30 % mean reversion, pre-tourney snapshot
  • Massey Ordinals   — POM (KenPom), SAG, COL, DOL, MOR, WLK, RTH
                        → converted to percentile ranks
  • Four Factors      — eFG%, TOV%, ORB%, FT-Rate (Dean Oliver)
  • Season Stats      — WinPct, AvgScore, AvgAllowed, Pace proxy
  • Strength-of-Sched — mean opponent Elo
  • Momentum          — last-10-game win rate (exponentially weighted)
  • Conference        — power-conf flag, season win-rate
  • Tournament Seed   — numeric seed (default 8.5 when unknown)
  • Coach Records     — career tourney win-rate (Men only)

Model Architecture
──────────────────
  • Separate pipelines for Men's and Women's data
  • LightGBM  +  XGBoost  (inverse-Brier weighted ensemble)
  • Time-based CV  →  train 2008–2022, validate 2023–2025
  • Isotonic calibration on validation residuals
  • Final predictions clipped to [0.025, 0.975]

Expected Performance
────────────────────
  Haupts (pure Elo baseline) : Men 0.187  Women 0.147
  Qasimi (LGB+XGB+CAT)       : 0.110  (tiny training set, no Massey)
  This pipeline target        : < 0.105  (full features + Massey)
================================================================
"""

# ================================================================
# SECTION 0 — Imports & Configuration
# ================================================================
import gc
import time
import warnings
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'], check=False)
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import brier_score_loss, log_loss
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)
t0 = time.time()

# ── Paths ────────────────────────────────────────────────────────
DATA   = Path('/kaggle/input/march-machine-learning-mania-2026')
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)

# ── Hyper-parameters & constants ────────────────────────────────
CFG = {
    # Elo
    'elo_k_regular'   : 20,       # K-factor for regular-season games
    'elo_k_tourney'   : 30,       # Higher K for tournament stakes
    'elo_init'        : 1500,     # Starting rating for every new team
    'elo_home_bonus'  : 75,       # Elo points added for true home games
    'elo_reversion'   : 0.30,     # 30 % mean-reversion between seasons
    'elo_margin_cap'  : 25,       # Cap margin of victory for K scaling

    # Massey
    'massey_systems'  : ['POM','SAG','COL','DOL','MOR','WLK','RTH'],
    'massey_day'      : 133,      # Pre-tourney snapshot day

    # Training
    'min_season'      : 2008,     # Allow 23 seasons of Elo burn-in
    'val_seasons'     : [2023, 2024, 2025],  # 3-season val — proven optimal
    'seed'            : 42,

    # Recency weighting — exponential decay applied per season.
    # A game from season S gets weight = recency_decay ^ (max_season - S).
    # decay=0.60 is the validated optimum (LB 0.04000 at this setting).
    # Validation seasons are always weighted 1.0 (they are recent by design).
    'recency_decay'   : 0.60,

    # Submission
    'clip_lo'         : 0.025,
    'clip_hi'         : 0.975,

    # Power conferences (affect conf-strength feature)
    'power_confs'     : {'acc','big_ten','big_twelve','sec','big_east',
                         'pac_twelve','wcc','a_ten'},
}

DIVIDER = '─' * 70
np.random.seed(CFG['seed'])

print(DIVIDER)
print('  March Machine Learning Mania 2026  —  Pipeline v2.4')
print(DIVIDER)


# ================================================================
# SECTION 1 — Data Loading
# ================================================================
print('\n[1/9] Loading data ...')

def load(fname: str) -> pd.DataFrame:
    """Load a competition CSV with a clean error message."""
    path = DATA / fname
    if not path.exists():
        raise FileNotFoundError(f'Missing: {path}')
    return pd.read_csv(path)

# Regular season — compact files only (small; kept in memory throughout)
m_reg_c  = load('MRegularSeasonCompactResults.csv')
w_reg_c  = load('WRegularSeasonCompactResults.csv')

# Tournament results — compact files only
m_tour_c = load('MNCAATourneyCompactResults.csv')
w_tour_c = load('WNCAATourneyCompactResults.csv')

# Detailed files are NOT loaded here.
# They are loaded lazily inside compute_four_factors() to avoid
# holding them in memory simultaneously with the 5.7M-row Massey table.
# File paths stored for lazy access:
M_REG_D_PATH = DATA / 'MRegularSeasonDetailedResults.csv'
W_REG_D_PATH = DATA / 'WRegularSeasonDetailedResults.csv'

# Seeds
m_seeds  = load('MNCAATourneySeeds.csv')
w_seeds  = load('WNCAATourneySeeds.csv')

# Teams
m_teams  = load('MTeams.csv')
w_teams  = load('WTeams.csv')

# Conferences
m_conf   = load('MTeamConferences.csv')
w_conf   = load('WTeamConferences.csv')

# Massey Ordinals (men only) — 5.7M rows; load once, free after feature build
massey   = load('MMasseyOrdinals.csv')

# Coaches (men only)
coaches  = load('MTeamCoaches.csv')

# Submission templates — load BOTH stages
# Stage 1: seasons 2022-2025 (519,144 rows) — active scoring window NOW
# Stage 2: season 2026       (132,133 rows) — scored after tourney starts
sub1 = load('SampleSubmissionStage1.csv')
sub2 = load('SampleSubmissionStage2.csv')

for _df in [sub1, sub2]:
    _df[['Season','T1','T2']] = (
        _df['ID'].str.split('_', expand=True).astype(int))

print(f'   Men  regular season : {len(m_reg_c):>7,} games (compact) + detailed loaded lazily')
print(f'   Women regular season: {len(w_reg_c):>7,} games (compact) + detailed loaded lazily')
print(f'   Men  tournament      : {len(m_tour_c):>7,} games')
print(f'   Women tournament     : {len(w_tour_c):>7,} games')
print(f'   Massey entries       : {len(massey):>7,}')
print(f'   Stage-1 matchups     : {len(sub1):>7,}  <- active submission window')
print(f'   Stage-2 matchups     : {len(sub2):>7,}  <- scored after tourney starts')


# ================================================================
# SECTION 2 — Elo Rating System
# ================================================================
print(f'\n[2/9] Computing Elo ratings ...')


def _expected(r_a: float, r_b: float) -> float:
    """Standard Elo expected score for team A."""
    return 1.0 / (1.0 + 10.0 ** ((r_b - r_a) / 400.0))


def _margin_multiplier(margin: int, cap: int = CFG['elo_margin_cap']) -> float:
    """
    Scale the K-factor by margin of victory.
    Uses log formula popularised by FiveThirtyEight:
        multiplier = ln(|margin| + 1) / ln(cap + 1)
    Capped at the provided maximum margin.
    """
    margin = min(abs(margin), cap)
    return np.log(margin + 1) / np.log(cap + 1)


def compute_elo(
    compact_df: pd.DataFrame,
    k_regular : int   = CFG['elo_k_regular'],
    k_tourney : int   = CFG['elo_k_tourney'],
    init      : float = CFG['elo_init'],
    home_bonus: float = CFG['elo_home_bonus'],
    reversion : float = CFG['elo_reversion'],
    tourney_day_start: int = 134,   # DayNum ≥ this → treat as tourney
) -> pd.DataFrame:
    """
    Compute end-of-regular-season Elo ratings for every team.

    Improvements over baseline Haupts implementation:
      - Margin-adjusted K-factor (log scale, capped)
      - Home-court bonus applied only to the expected-score calculation
        (clean stored ratings, no sign-flip bugs)
      - Tournament games use a higher K
      - Pre-tournament snapshot captured separately

    Returns
    -------
    DataFrame: [Season, TeamID, EloPreTourney, EloEndSeason]
    """
    ratings: Dict[int, float] = {}
    pre_tour_snapshots: List[dict] = []
    end_season_snapshots: List[dict] = []

    df = compact_df.sort_values(['Season', 'DayNum']).reset_index(drop=True)

    for season, grp in df.groupby('Season'):
        # ── Mean reversion at season start ───────────────────
        for tid in list(ratings.keys()):
            ratings[tid] = ratings[tid] + reversion * (init - ratings[tid])

        pre_tourney_saved = False

        for _, row in grp.iterrows():
            day   = row['DayNum']
            wid   = row['WTeamID']
            lid   = row['LTeamID']
            loc   = row.get('WLoc', 'N')
            score_diff = row['WScore'] - row['LScore']

            # ── Save pre-tourney snapshot once ───────────────
            if day >= tourney_day_start and not pre_tourney_saved:
                for tid, elo in ratings.items():
                    pre_tour_snapshots.append(
                        {'Season': season, 'TeamID': tid,
                         'EloPreTourney': elo})
                pre_tourney_saved = True

            # ── Initialise new teams ──────────────────────────
            if wid not in ratings: ratings[wid] = init
            if lid not in ratings: ratings[lid] = init

            r_w = ratings[wid]
            r_l = ratings[lid]

            # ── Home-court adjusted expected scores ──────────
            # Bonus is only applied to the probability estimate,
            # NOT stored in the rating itself → no sign-flip bug
            if loc == 'H':
                exp_w = _expected(r_w + home_bonus, r_l)
            elif loc == 'A':
                exp_w = _expected(r_w, r_l + home_bonus)
            else:
                exp_w = _expected(r_w, r_l)

            # ── K selection + margin multiplier ──────────────
            k_base = k_tourney if day >= tourney_day_start else k_regular
            k_adj  = k_base * _margin_multiplier(score_diff)

            delta         = k_adj * (1.0 - exp_w)   # winner won: actual=1
            ratings[wid]  = r_w + delta
            ratings[lid]  = r_l - delta

        # ── End-of-season snapshot ────────────────────────────
        # If season had no tourney games, still save pre-tourney snapshot
        if not pre_tourney_saved:
            for tid, elo in ratings.items():
                pre_tour_snapshots.append(
                    {'Season': season, 'TeamID': tid,
                     'EloPreTourney': elo})

        for tid, elo in ratings.items():
            end_season_snapshots.append(
                {'Season': season, 'TeamID': tid, 'EloEndSeason': elo})

    pre_df = pd.DataFrame(pre_tour_snapshots)
    end_df = pd.DataFrame(end_season_snapshots)

    merged = pre_df.merge(end_df, on=['Season', 'TeamID'], how='outer')
    # Fill pre-tourney with end-season where missing (edge years)
    merged['EloPreTourney'] = merged['EloPreTourney'].fillna(
        merged['EloEndSeason'])
    return merged


# Combine regular + tourney for full Elo trajectory
m_all_c = pd.concat([m_reg_c, m_tour_c], ignore_index=True)
w_all_c = pd.concat([w_reg_c, w_tour_c], ignore_index=True)

m_elo_df = compute_elo(m_all_c)
w_elo_df = compute_elo(w_all_c)

print(f'   Men  Elo table: {len(m_elo_df):,} team-seasons')
print(f'   Women Elo table: {len(w_elo_df):,} team-seasons')

# Fast lookup: (season, team_id) → elo
m_elo_lu: Dict[Tuple, float] = (
    m_elo_df.set_index(['Season','TeamID'])['EloPreTourney'].to_dict())
w_elo_lu: Dict[Tuple, float] = (
    w_elo_df.set_index(['Season','TeamID'])['EloPreTourney'].to_dict())


# ================================================================
# SECTION 3 — Massey Ordinals Features  (Men only)
# ================================================================
print(f'\n[3/9] Building Massey Ordinals features ...')

TARGET_SYSTEMS = CFG['massey_systems']


def build_massey_features(massey_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each (Season, TeamID), extract pre-tournament rankings from
    the 7 target systems and convert to percentile (higher = better).

    Steps
    -----
    1. Keep only DayNum ≤ massey_day (pre-tourney) → latest per system
    2. Pivot to wide format: one column per system
    3. For each system+season, convert rank to percentile
       percentile = (N_teams - rank) / (N_teams - 1)   ∈ [0,1]
    4. Compute mean_pct, min_pct (worst system), std_pct (consistency)
    """
    day_limit = CFG['massey_day']

    # Latest ranking per (Season, SystemName, TeamID) up to day_limit
    pre = massey_df[massey_df['RankingDayNum'] <= day_limit].copy()
    idx = (pre.groupby(['Season','SystemName','TeamID'])['RankingDayNum']
               .idxmax())
    pre = pre.loc[idx.values].reset_index(drop=True)

    # Keep only desired systems
    pre = pre[pre['SystemName'].isin(TARGET_SYSTEMS)]

    # Pivot: rows=(Season,TeamID), cols=system names
    pivot = pre.pivot_table(
        index=['Season','TeamID'],
        columns='SystemName',
        values='OrdinalRank',
        aggfunc='first',
    ).reset_index()
    pivot.columns.name = None
    sys_cols = [c for c in TARGET_SYSTEMS if c in pivot.columns]

    # Convert each rank to percentile within (Season, system)
    for sc in sys_cols:
        if sc not in pivot.columns:
            continue
        pivot[sc] = pivot.groupby('Season')[sc].transform(
            lambda x: (x.max() - x + 1) / x.count()   # higher = better
        )
        pivot.rename(columns={sc: f'mas_{sc}'}, inplace=True)

    pct_cols = [f'mas_{s}' for s in sys_cols if f'mas_{s}' in pivot.columns]

    pivot['mas_mean']   = pivot[pct_cols].mean(axis=1)
    pivot['mas_min']    = pivot[pct_cols].min(axis=1)    # worst ranking
    pivot['mas_std']    = pivot[pct_cols].std(axis=1)    # consistency
    pivot['mas_n_sys']  = pivot[pct_cols].notna().sum(axis=1)

    print(f'   Massey: {len(pivot):,} team-seasons '
          f'| systems: {sys_cols}')
    return pivot


massey_feats = build_massey_features(massey)
massey_lu: Dict[Tuple, dict] = (
    massey_feats.set_index(['Season','TeamID']).to_dict('index'))

MAS_COLS = [c for c in massey_feats.columns
            if c not in ('Season','TeamID')]

# Free the large Massey source DF — no longer needed
del massey_feats
del massey        # 5.7M row source no longer needed
gc.collect()


# ================================================================
# SECTION 4 — Four Factors & Box-Score Features
# ================================================================
print(f'\n[4/9] Computing Four Factors & box-score features ...')


def compute_four_factors(detail_path: Path) -> pd.DataFrame:
    """
    Compute Dean Oliver's Four Factors + extended stats per team-season.

    Accepts a FILE PATH (not a DataFrame) so the detailed CSV is loaded
    inside this function's local scope, processed, and immediately freed
    before returning — preventing it from co-existing in memory with the
    5.7M-row Massey table that was freed just before this is called.

    Only the 20 required columns are read from disk (usecols) to further
    reduce peak memory.  Fully vectorised — zero Python-level loops.

    Four Factors (Dean Oliver)
    ──────────────────────────
      eFG%   = (FGM + 0.5·FGM3) / FGA        higher = better
      TOV%   = TO / (FGA + 0.44·FTA + TO)    lower  = better
      ORB%   = OR / (OR + Opp_DR)             higher = better
      FTRate = FTM / FGA                      higher = better
    """
    # ── 1. Load only needed columns, filter to regular season ────
    need = ['Season','DayNum',
            'WTeamID','WScore','WFGM','WFGA','WFGM3','WFGA3',
            'WFTM','WFTA','WOR','WDR','WAst','WTO',
            'LTeamID','LScore','LFGM','LFGA','LFGM3','LFGA3',
            'LFTM','LFTA','LOR','LDR','LAst','LTO']

    # Read only existing columns to handle files that may lack some stats
    all_cols = pd.read_csv(detail_path, nrows=0).columns.tolist()
    use_cols = [c for c in need if c in all_cols]

    df = pd.read_csv(detail_path, usecols=use_cols,
                     dtype={c: 'float32' for c in use_cols
                            if c not in ('Season','DayNum','WTeamID','LTeamID')})

    # Regular season only
    df = df[df['DayNum'] <= 132].copy()

    # ── 2. Winner-side DataFrame (vectorised rename) ──────────
    w_side = pd.DataFrame({
        'Season' : df['Season'],
        'TeamID' : df['WTeamID'],
        'Win'    : 1,
        'Score'  : df['WScore'],
        'Allowed': df['LScore'],
        'FGM'    : df['WFGM'],  'FGA'  : df['WFGA'],
        'FGM3'   : df['WFGM3'], 'FGA3' : df['WFGA3'],
        'FTM'    : df['WFTM'],  'FTA'  : df['WFTA'],
        'OR'     : df['WOR'],   'DR'   : df['WDR'],
        'Ast'    : df['WAst'],  'TO'   : df['WTO'],
        'OppDR'  : df['LDR'],   # opponent defensive boards → for ORB%
    })

    # ── 3. Loser-side DataFrame (vectorised rename) ───────────
    l_side = pd.DataFrame({
        'Season' : df['Season'],
        'TeamID' : df['LTeamID'],
        'Win'    : 0,
        'Score'  : df['LScore'],
        'Allowed': df['WScore'],
        'FGM'    : df['LFGM'],  'FGA'  : df['LFGA'],
        'FGM3'   : df['LFGM3'], 'FGA3' : df['LFGA3'],
        'FTM'    : df['LFTM'],  'FTA'  : df['LFTA'],
        'OR'     : df['LOR'],   'DR'   : df['LDR'],
        'Ast'    : df['LAst'],  'TO'   : df['LTO'],
        'OppDR'  : df['WDR'],
    })

    # ── 4. Free the filtered source DF before concat ──────────
    del df
    gc.collect()

    # ── 5. Stack and aggregate ────────────────────────────────
    g = pd.concat([w_side, l_side], ignore_index=True)
    del w_side, l_side
    gc.collect()

    sum_cols = ['Win','Score','Allowed','FGM','FGA','FGM3','FGA3',
                'FTM','FTA','OR','DR','Ast','TO','OppDR']

    agg_dict = {c: (c, 'sum') for c in sum_cols}
    agg_dict['Games'] = ('Win', 'count')
    agg = g.groupby(['Season','TeamID']).agg(**agg_dict).reset_index()

    # ── Std features (need per-game data still in g) ──────────
    # MarginStd (#2 feature in reference) — consistency/volatility signal
    # FTRateStd (#3 feature) — free throw rate variance
    eps_g = 1e-6
    g['Margin']  = g['Score'] - g['Allowed']
    g['FTRateG'] = g['FTM'] / (g['FGA'] + eps_g)
    std_agg = g.groupby(['Season','TeamID']).agg(
        MarginStd  = ('Margin',  'std'),
        FTRateStd  = ('FTRateG', 'std'),
    ).reset_index().fillna(0)

    agg = agg.merge(std_agg, on=['Season','TeamID'], how='left')
    del g, std_agg
    gc.collect()

    # ── 6. Derived features ───────────────────────────────────
    eps = 1e-6

    agg['WinPct']    = agg['Win']   / agg['Games']
    agg['AvgScore']  = agg['Score'] / agg['Games']
    agg['AvgAllow']  = agg['Allowed'] / agg['Games']
    agg['AvgMargin'] = agg['AvgScore'] - agg['AvgAllow']

    # Four Factors
    agg['eFGPct']  = (agg['FGM'] + 0.5*agg['FGM3']) / (agg['FGA'] + eps)
    agg['TOVPct']  = agg['TO'] / (agg['FGA'] + 0.44*agg['FTA'] + agg['TO'] + eps)
    agg['ORBPct']  = agg['OR'] / (agg['OR'] + agg['OppDR'] + eps)
    agg['FTRate']  = agg['FTM'] / (agg['FGA'] + eps)

    # Extended
    agg['FGPct']   = agg['FGM']  / (agg['FGA']  + eps)
    agg['FG3Pct']  = agg['FGM3'] / (agg['FGA3'] + eps)
    agg['AstTORat']= agg['Ast']  / (agg['TO']   + eps)

    # Keep only derived + index columns
    keep_final = ['Season','TeamID','Games','WinPct',
                  'AvgScore','AvgAllow','AvgMargin','MarginStd','FTRateStd',
                  'eFGPct','TOVPct','ORBPct','FTRate',
                  'FGPct','FG3Pct','AstTORat']
    agg = agg[keep_final]

    print(f'   Four Factors: {len(agg):,} team-seasons '
          f'| {agg.shape[1]-2} features each')
    return agg


print("   Loading Men's detailed (lazy, usecols only) ...")
m_ff = compute_four_factors(M_REG_D_PATH)
gc.collect()
print("   Loading Women's detailed (lazy, usecols only) ...")
w_ff = compute_four_factors(W_REG_D_PATH)
gc.collect()

m_ff_lu = m_ff.set_index(['Season','TeamID']).to_dict('index')
w_ff_lu = w_ff.set_index(['Season','TeamID']).to_dict('index')

FF_COLS = [c for c in m_ff.columns if c not in ('Season','TeamID')]


# ================================================================
# SECTION 5 — SOS, Momentum, Conference, Seed, Coach features
# ================================================================
print(f'\n[5/9] Building auxiliary features ...')


# ── 5a: Strength of Schedule (mean opponent Elo) ─────────────────
def compute_sos(compact_df: pd.DataFrame,
                elo_lu: Dict) -> Dict[Tuple, float]:
    """
    Average pre-tournament Elo of all opponents faced by each team
    in the regular season of that season.
    Uses vectorised merge — no iterrows.
    """
    df = compact_df[compact_df['DayNum'] <= 132].copy()

    # Winner perspective: opponent = Loser
    w_side = df[['Season','WTeamID','LTeamID']].rename(
        columns={'WTeamID':'TeamID','LTeamID':'OppID'})
    # Loser perspective: opponent = Winner
    l_side = df[['Season','LTeamID','WTeamID']].rename(
        columns={'LTeamID':'TeamID','WTeamID':'OppID'})

    both = pd.concat([w_side, l_side], ignore_index=True)

    # Vectorised lookup — build a Series keyed by (Season, OppID) tuple
    both['_key']    = list(zip(both['Season'], both['OppID']))
    both['OppElo']  = both['_key'].map(elo_lu).fillna(CFG['elo_init'])
    both.drop(columns=['_key'], inplace=True)

    sos = (both.groupby(['Season','TeamID'])['OppElo']
               .mean().reset_index(name='SOS'))
    return sos.set_index(['Season','TeamID'])['SOS'].to_dict()


m_sos_lu = compute_sos(m_reg_c, m_elo_lu)
w_sos_lu = compute_sos(w_reg_c, w_elo_lu)
print(f'   SOS: Men {len(m_sos_lu):,} | Women {len(w_sos_lu):,}')


# ── 5b: Momentum — exponentially weighted last-10 win rate ────────
def compute_momentum(compact_df: pd.DataFrame) -> Dict[Tuple, float]:
    """
    For each (Season, TeamID), compute the exponentially-weighted
    win rate over the last 10 regular-season games.
    Games closer to the tournament have higher weight (decay=0.85).

    Vectorised implementation:
    ─────────────────────────
    1. Build a tall table with one row per (game, team) — winner=1, loser=0
    2. Sort by (Season, TeamID, DayNum)
    3. Use groupby cumcount to get game-sequence index
    4. Keep only the last 10 rows per group
    5. Apply exponential weights via a pre-computed lookup
    """
    df = compact_df[compact_df['DayNum'] <= 132].copy()

    # ── Stack winner / loser into one long table ──────────────
    w_side = df[['Season','DayNum','WTeamID']].rename(
        columns={'WTeamID':'TeamID'})
    w_side['Win'] = 1

    l_side = df[['Season','DayNum','LTeamID']].rename(
        columns={'LTeamID':'TeamID'})
    l_side['Win'] = 0

    tall = pd.concat([w_side, l_side], ignore_index=True)
    tall.sort_values(['Season','TeamID','DayNum'], inplace=True)
    tall.reset_index(drop=True, inplace=True)

    # ── Reverse-game index within each group (0 = most recent) ─
    tall['RevIdx'] = tall.groupby(['Season','TeamID']).cumcount(ascending=False)

    # Keep only last 10 games per team-season
    last10 = tall[tall['RevIdx'] < 10].copy()

    # Exponential weight: more recent = higher weight
    decay   = 0.85
    last10['Weight'] = decay ** last10['RevIdx']

    # Weighted win rate = sum(Win * Weight) / sum(Weight) per group
    grp = last10.groupby(['Season','TeamID'])
    wsum  = grp.apply(lambda x: (x['Win'] * x['Weight']).sum())
    dsum  = grp['Weight'].sum()
    mom   = (wsum / dsum).reset_index()
    mom.columns = ['Season','TeamID','Momentum']

    del tall, last10, w_side, l_side
    gc.collect()

    return mom.set_index(['Season','TeamID'])['Momentum'].to_dict()


m_mom_lu = compute_momentum(m_reg_c)
w_mom_lu = compute_momentum(w_reg_c)
print(f'   Momentum: Men {len(m_mom_lu):,} | Women {len(w_mom_lu):,}')


# ── 5c: Conference features ────────────────────────────────────────
def compute_conf_features(compact_df: pd.DataFrame,
                          conf_df: pd.DataFrame
                          ) -> Tuple[Dict, Dict]:
    """
    Returns two dicts:
      conf_lu      : (season, team_id) → conference abbreviation
      conf_wr_lu   : (season, conf)    → season win-rate
    """
    conf_lu = (conf_df.set_index(['Season','TeamID'])['ConfAbbrev'].to_dict())

    w_conf = compact_df.merge(
        conf_df.rename(columns={'TeamID':'WTeamID','ConfAbbrev':'WConf'}),
        on=['Season','WTeamID'], how='left')
    l_conf = compact_df.merge(
        conf_df.rename(columns={'TeamID':'LTeamID','ConfAbbrev':'LConf'}),
        on=['Season','LTeamID'], how='left')

    ww = w_conf.groupby(['Season','WConf']).size().reset_index(name='W')
    ll = l_conf.groupby(['Season','LConf']).size().reset_index(name='L')
    ww.rename(columns={'WConf':'Conf'}, inplace=True)
    ll.rename(columns={'LConf':'Conf'}, inplace=True)

    cdf = ww.merge(ll, on=['Season','Conf'], how='outer').fillna(0)
    cdf['WinRate'] = cdf['W'] / (cdf['W'] + cdf['L'] + 1e-6)
    conf_wr_lu = cdf.set_index(['Season','Conf'])['WinRate'].to_dict()

    return conf_lu, conf_wr_lu


m_conf_lu, m_conf_wr_lu = compute_conf_features(m_reg_c, m_conf)
w_conf_lu, w_conf_wr_lu = compute_conf_features(w_reg_c, w_conf)
print(f'   Conference: Men {len(m_conf_wr_lu):,} | Women {len(w_conf_wr_lu):,}')


# ── 5d: Tournament seeds ──────────────────────────────────────────
def build_seed_lu(seeds_df: pd.DataFrame) -> Dict[Tuple, float]:
    """Convert seed string (e.g. 'W01') to integer, store in dict."""
    df = seeds_df.copy()
    df['SeedNum'] = df['Seed'].str.extract(r'(\d+)').astype(int)
    return df.set_index(['Season','TeamID'])['SeedNum'].to_dict()


m_seed_lu = build_seed_lu(m_seeds)
w_seed_lu = build_seed_lu(w_seeds)


# ── 5e: Coach tournament records (Men only) ────────────────────────
def compute_coach_tourney_records(
        coaches_df: pd.DataFrame,
        tourney_df: pd.DataFrame) -> Dict[Tuple, float]:
    """
    For each (Season, TeamID), return the head coach's career
    tournament win-rate going INTO that season (no leakage).
    """
    # Keep only end-of-season coach (LastDayNum ≥ 132)
    active = coaches_df[coaches_df['LastDayNum'] >= 132].copy()

    coach_season = active[['Season','TeamID','CoachName']].drop_duplicates(
        subset=['Season','TeamID'])

    # Merge win/loss into historical tourney
    tw = tourney_df.merge(
        coach_season.rename(columns={'TeamID':'WTeamID','CoachName':'WCoach'}),
        on=['Season','WTeamID'], how='left')
    tl = tourney_df.merge(
        coach_season.rename(columns={'TeamID':'LTeamID','CoachName':'LCoach'}),
        on=['Season','LTeamID'], how='left')

    coach_wins   = tw.groupby('WCoach').size().reset_index(name='TWins')
    coach_losses = tl.groupby('LCoach').size().reset_index(name='TLosses')
    coach_wins.rename(columns={'WCoach':'Coach'}, inplace=True)
    coach_losses.rename(columns={'LCoach':'Coach'}, inplace=True)

    crec = coach_wins.merge(coach_losses, on='Coach', how='outer').fillna(0)
    crec['CoachTWR'] = crec['TWins'] / (crec['TWins'] + crec['TLosses'] + 1e-6)

    # Map back: (season, team_id) → coach → career tourney win-rate
    coach_twr = crec.set_index('Coach')['CoachTWR'].to_dict()
    coach_map = coach_season.set_index(['Season','TeamID'])['CoachName'].to_dict()

    result = {}
    for (s, tid), cname in coach_map.items():
        result[(s, tid)] = coach_twr.get(cname, 0.5)
    return result


m_coach_lu = compute_coach_tourney_records(coaches, m_tour_c)
print(f'   Coach records: {len(m_coach_lu):,} team-seasons')


# ── 5f: Head-to-Head tournament history ─────────────────────────────
def compute_h2h(m_tourney: pd.DataFrame,
                w_tourney: pd.DataFrame) -> Dict[Tuple, float]:
    """
    For each canonical (T1, T2) pair where T1 < T2, compute their
    historical tournament head-to-head win rate for T1.

    Bayesian shrinkage toward 0.5 is applied to avoid over-trusting
    small samples (e.g. only 1-2 prior matchups):
        shrunk_rate = (n_wins + prior_n * 0.5) / (n_games + prior_n)

    This is the #1 feature in the reference implementation (importance
    578 vs 221 for the #2 feature), because tournament experience between
    specific programmes carries real predictive signal beyond ratings.

    Teams with no prior matchup default to 0.5 (no information).
    """
    prior_n = 3   # Bayesian prior equivalent to 3 games at 50%

    all_tourney = pd.concat([m_tourney, w_tourney], ignore_index=True)
    all_tourney = all_tourney[['Season','WTeamID','LTeamID']].copy()

    # Canonical ordering
    all_tourney['T1']    = all_tourney[['WTeamID','LTeamID']].min(axis=1)
    all_tourney['T2']    = all_tourney[['WTeamID','LTeamID']].max(axis=1)
    all_tourney['T1Win'] = (all_tourney['WTeamID'] == all_tourney['T1']).astype(int)

    grp = all_tourney.groupby(['T1','T2'])['T1Win'].agg(['sum','count'])
    grp.columns = ['Wins','Games']

    # Bayesian shrinkage: pull win rate toward 0.5 for small samples
    grp['H2H'] = (grp['Wins'] + prior_n * 0.5) / (grp['Games'] + prior_n)

    # Return two lookups: win rate and game count
    win_lu   = grp['H2H'].to_dict()
    games_lu = grp['Games'].clip(upper=5).to_dict()   # cap at 5 per reference
    print(f'   H2H: {len(win_lu):,} unique matchup pairs (prior_n={prior_n})')
    return win_lu, games_lu   # key: (t1_id, t2_id) — t1 < t2


h2h_lu, h2h_games_lu = compute_h2h(m_tour_c, w_tour_c)


# ── 5g: Tournament experience features ──────────────────────────
def compute_tourney_experience(
        m_tourney : pd.DataFrame,
        w_tourney : pd.DataFrame,
        m_seeds   : pd.DataFrame,
        w_seeds   : pd.DataFrame,
        window    : int = 6,
) -> Dict[Tuple, dict]:
    """
    For each (season, team_id), compute:
        TourneyApps  — appearances in last `window` seasons
        TourneyWins  — avg tournament wins per appearance (last window)
        DeepRunRate  — fraction of appearances reaching Sweet 16+ (round >= 3)
        SeedAvg      — avg seed over last window appearances

    These capture programme-level tournament quality that Elo only
    partially reflects — e.g. a team that consistently reaches the
    Elite Eight but gets upset early this year is still dangerous.
    """
    # Combine men's + women's, tracking wins per team per season
    all_t   = pd.concat([
        m_tourney[['Season','WTeamID','LTeamID']],
        w_tourney[['Season','WTeamID','LTeamID']],
    ], ignore_index=True)

    # Count wins per (season, team)
    wins = all_t.groupby(['Season','WTeamID']).size().reset_index()
    wins.columns = ['Season','TeamID','Wins']

    # Count appearances per (season, team) — appear if in seeds
    all_seeds = pd.concat([
        m_seeds[['Season','TeamID','Seed']],
        w_seeds[['Season','TeamID','Seed']],
    ], ignore_index=True)
    all_seeds['SeedNum'] = all_seeds['Seed'].str[1:3].astype(int)

    # Merge wins into seeds (0 wins if no wins that season)
    apps = all_seeds.merge(wins, on=['Season','TeamID'], how='left').fillna({'Wins': 0})
    apps['Wins'] = apps['Wins'].astype(int)
    # Deep run = 2+ wins (Sweet 16 or better)
    apps['DeepRun'] = (apps['Wins'] >= 2).astype(int)

    lu = {}
    all_seasons = sorted(apps['Season'].unique())
    for season in all_seasons:
        past = apps[apps['Season'].between(season - window, season - 1)]
        for team_id, grp in past.groupby('TeamID'):
            n_apps = len(grp)
            lu[(season, int(team_id))] = {
                'TourneyApps'  : n_apps,
                'TourneyWPG'   : grp['Wins'].mean() if n_apps else 0.0,
                'DeepRunRate'  : grp['DeepRun'].mean() if n_apps else 0.0,
                'SeedAvg'      : grp['SeedNum'].mean() if n_apps else 8.5,
            }

    n_teams = len({k[1] for k in lu})
    print(f'   TourneyExp: {len(lu):,} (season, team) records  |  {n_teams:,} teams')
    return lu


tourney_exp_lu = compute_tourney_experience(
    m_tour_c, w_tour_c, m_seeds, w_seeds
)




# ================================================================
# SECTION 6 — Feature Assembly
# ================================================================
print(f'\n[6/9] Assembling feature matrix ...')


def get_team_features(
        season   : int,
        team_id  : int,
        elo_lu   : Dict,
        ff_lu    : Dict,
        sos_lu   : Dict,
        mom_lu   : Dict,
        seed_lu  : Dict,
        conf_lu  : Dict,
        conf_wr_lu: Dict,
        mas_lu   : Optional[Dict]   = None,
        coach_lu : Optional[Dict]   = None,
        init_elo : float            = CFG['elo_init'],
) -> dict:
    """
    Return a flat feature dict for a single (season, team_id) lookup.
    All lookups are O(1) dict.get() with sensible defaults.
    """
    elo  = elo_lu.get((season, team_id), init_elo)
    ff   = ff_lu.get((season, team_id), {})
    sos  = sos_lu.get((season, team_id), init_elo)
    mom  = mom_lu.get((season, team_id), 0.5)
    seed = seed_lu.get((season, team_id), 8.5)   # unknown → average seed

    conf    = conf_lu.get((season, team_id), 'unk')
    conf_wr = conf_wr_lu.get((season, conf), 0.5)
    is_pwr  = 1 if conf in CFG['power_confs'] else 0

    coach_twr = coach_lu.get((season, team_id), 0.5) if coach_lu else 0.5

    # Pythagorean expectation (Bill James / Daryl Morey NCAA version)
    # Uses exponent 11.5 — proven optimal for college basketball.
    # Captures scoring efficiency better than raw win% for small samples.
    avg_sc  = ff.get('AvgScore', 70.0)
    avg_all = ff.get('AvgAllow', 70.0)
    _e      = 1e-6
    pyth    = avg_sc**11.5 / (avg_sc**11.5 + avg_all**11.5 + _e)

    feats = {
        # Elo
        'Elo'        : elo,
        # Four Factors & shooting
        'WinPct'     : ff.get('WinPct',    0.5),
        'AvgScore'   : avg_sc,
        'AvgAllowed' : avg_all,
        'AvgMargin'  : ff.get('AvgMargin',  0.0),
        'MarginStd'  : ff.get('MarginStd',  8.0),   # ← #2 feature in reference
        'eFGPct'     : ff.get('eFGPct',   0.490),
        'TOVPct'     : ff.get('TOVPct',   0.170),
        'ORBPct'     : ff.get('ORBPct',   0.300),
        'FTRate'     : ff.get('FTRate',   0.240),
        'FTRateStd'  : ff.get('FTRateStd',0.060),   # ← #3 feature in reference
        'FGPct'      : ff.get('FGPct',    0.440),
        'FG3Pct'     : ff.get('FG3Pct',   0.330),
        'AstTORat'   : ff.get('AstTORat', 1.100),
        'Pyth'       : pyth,                         # ← #11 feature in reference
        # Context
        'SOS'        : sos,
        'Momentum'   : mom,
        'Seed'       : seed,
        'ConfWR'     : conf_wr,
        'IsPower'    : is_pwr,
        'CoachTWR'   : coach_twr,
    }

    # Tournament experience (both genders)
    exp = tourney_exp_lu.get((season, team_id), {})
    feats['TourneyApps']  = exp.get('TourneyApps',  0.0)
    feats['TourneyWPG']   = exp.get('TourneyWPG',   0.0)
    feats['DeepRunRate']  = exp.get('DeepRunRate',   0.0)
    feats['SeedAvg']      = exp.get('SeedAvg',       8.5)

    # Massey (men only; NaN-fill for women)
    if mas_lu is not None:
        m = mas_lu.get((season, team_id), {})
        feats['MasMean'] = m.get('mas_mean', 0.5)
        feats['MasMin']  = m.get('mas_min',  0.5)
        feats['MasStd']  = m.get('mas_std',  0.1)
        feats['MasPOM']  = m.get('mas_POM',  0.5)
        feats['MasSAG']  = m.get('mas_SAG',  0.5)
    else:
        feats['MasMean'] = 0.5
        feats['MasMin']  = 0.5
        feats['MasStd']  = 0.1
        feats['MasPOM']  = 0.5
        feats['MasSAG']  = 0.5

    return feats


TEAM_FEAT_NAMES = list(get_team_features(
    2020, 1234, m_elo_lu, m_ff_lu, m_sos_lu, m_mom_lu,
    m_seed_lu, m_conf_lu, m_conf_wr_lu, massey_lu, m_coach_lu
).keys())


def build_matchup_features(
        season   : int,
        t1_id    : int,
        t2_id    : int,
        is_women : bool,
) -> dict:
    """
    Build a single matchup feature vector.
    t1_id < t2_id  (canonical lower-ID = Team1 convention).
    Returns per-team features + differentials (T1 − T2).
    """
    if is_women:
        elo_lu  = w_elo_lu;  ff_lu  = w_ff_lu
        sos_lu  = w_sos_lu;  mom_lu = w_mom_lu
        seed_lu = w_seed_lu; conf_lu = w_conf_lu; conf_wr = w_conf_wr_lu
        mas_lu  = None;      coach_lu = None
    else:
        elo_lu  = m_elo_lu;  ff_lu  = m_ff_lu
        sos_lu  = m_sos_lu;  mom_lu = m_mom_lu
        seed_lu = m_seed_lu; conf_lu = m_conf_lu; conf_wr = m_conf_wr_lu
        mas_lu  = massey_lu; coach_lu = m_coach_lu

    f1 = get_team_features(season, t1_id, elo_lu, ff_lu, sos_lu, mom_lu,
                           seed_lu, conf_lu, conf_wr, mas_lu, coach_lu)
    f2 = get_team_features(season, t2_id, elo_lu, ff_lu, sos_lu, mom_lu,
                           seed_lu, conf_lu, conf_wr, mas_lu, coach_lu)

    row = {}
    for k in TEAM_FEAT_NAMES:
        row[f'T1_{k}'] = f1[k]
        row[f'T2_{k}'] = f2[k]
        row[f'D_{k}']  = f1[k] - f2[k]   # differential (most informative)

    # ── H2H: historical tournament head-to-head ───────────────────
    # t1_id < t2_id by convention, so the key is always (t1_id, t2_id)
    # and the value is T1's shrunk win rate (0.5 = no history)
    h2h_key = (t1_id, t2_id)
    row['H2H_WinRate'] = h2h_lu.get(h2h_key, 0.5)
    row['H2H_Games']   = h2h_games_lu.get(h2h_key, 0)   # #15 in reference

    # ── Interaction features (from reference: top signal combos) ──
    # These capture non-linear joint effects that tree models can
    # learn independently, but explicit features speed up learning
    # and help with our small tournament-only training set.
    elo_diff   = row['D_Elo']
    seed_diff  = row['D_Seed']
    net_diff   = row['D_AvgMargin']   # proxy for NetRtg diff
    pyth_diff  = row['D_Pyth']

    row['IX_Elo_x_Seed']    = elo_diff  * seed_diff    # ← reference #1 interaction
    row['IX_Net_x_Seed']    = net_diff  * seed_diff    # ← reference #2 interaction
    row['IX_Elo_x_Net']     = elo_diff  * net_diff     # ← reference #3 interaction
    row['IX_Seed_x_Pyth']   = seed_diff * pyth_diff    # ← reference #4 interaction
    row['IX_Off_x_Def']     = row['D_eFGPct'] * (-row['D_TOVPct'])  # ← ref #6
    row['EloWinProb']       = 1.0 / (1.0 + 10 ** (-elo_diff / 400.0))
    row['SeedWinProb']      = 1.0 / (1.0 + np.exp(0.4 * seed_diff))
    return row


# Verify feature count
sample_row = build_matchup_features(2024, 1101, 1200, is_women=False)
N_FEATS = len(sample_row)
FEAT_COLS = list(sample_row.keys())
print(f'   Feature vector: {N_FEATS} features per matchup')
print(f'   ({len(TEAM_FEAT_NAMES)} team features × 3 views: T1 / T2 / Diff)')


# ================================================================
# SECTION 7 — Training Dataset Construction
# ================================================================
print(f'\n[7/9] Building training dataset ...')


def build_training_set(
        tourney_df : pd.DataFrame,
        is_women   : bool,
        min_season : int = CFG['min_season'],
) -> pd.DataFrame:
    """
    Convert historical tournament results into (features, label) pairs.

    Label = 1 if the lower-TeamID team won, else 0.
    This matches the submission file convention.
    """
    df   = tourney_df[tourney_df['Season'] >= min_season].copy()
    rows = []

    for _, r in tqdm(df.iterrows(), total=len(df),
                     desc=f"  {'Women' if is_women else 'Men ':5s} tourney rows"):
        s, wid, lid = r['Season'], r['WTeamID'], r['LTeamID']

        # Canonical ordering: lower ID = Team1
        if wid < lid:
            t1, t2, outcome = wid, lid, 1
        else:
            t1, t2, outcome = lid, wid, 0

        feats = build_matchup_features(s, t1, t2, is_women)
        feats['Outcome'] = outcome
        feats['Season']  = s
        rows.append(feats)

    result = pd.DataFrame(rows)
    del rows
    gc.collect()
    return result


m_train = build_training_set(m_tour_c, is_women=False)
w_train = build_training_set(w_tour_c, is_women=True)

# Combine with gender flag
m_train['IsWomen'] = 0
w_train['IsWomen'] = 1
train_base = pd.concat([m_train, w_train], ignore_index=True)

# ── Symmetric augmentation — eliminate T1/T2 ordering bias ───────
# The submission always puts lower TeamID as T1.  Without augmentation,
# the model can learn spurious correlations tied to team-ID magnitude
# (e.g. older programmes with lower IDs tend to be stronger).
# Fix: add every matchup in reverse (T2→T1, T1→T2) with flipped outcome.
# This makes the model blind to which team is T1 vs T2 by construction,
# and is why the prediction mean should converge to exactly 0.50.
#
# Implementation: swap all T1_* ↔ T2_* columns; negate all D_* columns;
# flip Outcome. Season and IsWomen stay the same.

t1_cols  = [c for c in train_base.columns if c.startswith('T1_')]
t2_cols  = [c for c in train_base.columns if c.startswith('T2_')]
d_cols   = [c for c in train_base.columns if c.startswith('D_')]

train_aug = train_base.copy()

# Swap T1 ↔ T2 values
train_aug[t1_cols] = train_base[t2_cols].values
train_aug[t2_cols] = train_base[t1_cols].values

# Flip all differentials (T1-T2 becomes T2-T1)
train_aug[d_cols]  = -train_base[d_cols].values

# Flip outcome: if original said T1 wins (1), augmented says T2 wins (0)
train_aug['Outcome'] = 1 - train_base['Outcome']

train_df = pd.concat([train_base, train_aug], ignore_index=True)
del train_base, train_aug
gc.collect()

print(f'\n   Base training rows   : {len(m_train) + len(w_train):,}')
print(f'   After augmentation   : {len(train_df):,} (2× — T1/T2 symmetric)')
print(f'   Men   ({CFG["min_season"]}–2025): {len(m_train):,} tournament games')
print(f'   Women ({CFG["min_season"]}–2025): {len(w_train):,} tournament games')
print(f'   Class balance        : {train_df["Outcome"].mean():.3f} '
      f'(should be exactly 0.50 after augmentation)')


# ================================================================
# SECTION 8 — Model Training  (Separate Men's & Women's models)
# ================================================================
# Why separate models?
# ───────────────────
# Our EDA showed the two tournaments have structurally different
# outcome distributions:
#   Men's  upset rate : 27.3%   Haupts Elo Brier : 0.187
#   Women's upset rate: 21.1%   Haupts Elo Brier : 0.147
#
# A single combined model with IsWomen=0/1 is forced to share its
# decision boundary across both distributions.  Two separate models
# can independently learn:
#   - Women's stronger favourite dominance (UConn effect, 0% 14/15-seeds)
#   - Men's higher upset probability at every seed tier
#   - Different optimal feature weights (Massey matters more for men;
#     seed gap matters more for women)
# ================================================================
print(f'\n[8/9] Training models (separate Men\'s & Women\'s) ...')


# ── Leakage-safe shadow feature selection ─────────────────────────
# Key insight from reference pipeline: selecting only ~15 features
# from 70+ dramatically reduces noise and prevents overfitting on
# our small (~2,200 row) training set.
#
# Method (Boruta-style shadow permutation):
#   1. Train a probe LGB on early seasons only (held_out=5)
#      so the selector never sees the validation seasons
#   2. Keep top-N features by importance
#   3. For each feature, compare its importance against the maximum
#      importance of its shuffled (shadow) version across n_runs
#   4. Features that beat their shadow ≥ 40% of the time are kept
#   5. Core domain features are always kept regardless
#
# This eliminates features that are just noise on this small dataset
# without discarding features with genuine predictive power.
def shadow_feature_selection(
        train_df    : pd.DataFrame,
        feat_cols   : list,
        gender      : str,
        val_seasons : list,
        n_top       : int = 60,
        n_runs      : int = 5,
        threshold   : float = 0.40,
        held_out    : int = 5,
) -> list:
    # Returns a reduced list of feature columns that beat their shadow.
    # Runs on early-season training data only — never touches val seasons.
    # Only use early seasons for selection (leakage-safe)
    safe_df  = train_df[~train_df['Season'].isin(val_seasons)].copy()
    unique_s = sorted(safe_df['Season'].unique())
    safe_df  = safe_df[safe_df['Season'].isin(unique_s[:-held_out])]

    if len(safe_df) < 100:
        print(f'   {gender} too few safe rows ({len(safe_df)}) — skipping selection')
        return feat_cols

    X_safe = safe_df[feat_cols].values.astype(np.float32)
    y_safe = safe_df['Outcome'].values

    # Fill NaNs with column median
    col_medians = np.nanmedian(X_safe, axis=0)
    nan_mask    = np.isnan(X_safe)
    X_safe[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])

    print(f'   {gender} feature selection on {len(safe_df):,} games '
          f'({len(unique_s)-held_out} early seasons) ...')

    # Step 1: probe importance on top-N
    probe = lgb.LGBMClassifier(
        n_estimators=300, num_leaves=31, max_depth=5,
        learning_rate=0.05, random_state=CFG['seed'], verbose=-1
    )
    probe.fit(X_safe, y_safe)
    imp = pd.Series(probe.feature_importances_, index=feat_cols)
    top_feats = imp.nlargest(n_top).index.tolist()
    top_idx   = [feat_cols.index(f) for f in top_feats]
    X_top     = X_safe[:, top_idx]

    # Step 2: shadow permutation test
    shadow_wins = np.zeros(len(top_feats))
    for run in range(n_runs):
        rng      = np.random.RandomState(CFG['seed'] + run)
        X_shadow = np.array([rng.permutation(X_top[:, j])
                              for j in range(X_top.shape[1])]).T
        X_aug    = np.hstack([X_top, X_shadow])
        probe2   = lgb.LGBMClassifier(
            n_estimators=200, num_leaves=15, max_depth=4,
            learning_rate=0.05, random_state=CFG['seed'] + run, verbose=-1
        )
        probe2.fit(X_aug, y_safe)
        imp2         = probe2.feature_importances_
        real_imp     = imp2[:len(top_feats)]
        shadow_max   = imp2[len(top_feats):].max()
        shadow_wins += (real_imp > shadow_max).astype(float)

    keep_mask = (shadow_wins / n_runs) >= threshold
    selected  = [f for f, k in zip(top_feats, keep_mask) if k]

    # Step 3: always keep core domain features
    must_keep = [
        'D_Elo', 'D_Seed', 'D_MasMean', 'D_MasPOM',
        'D_WinPct', 'D_AvgMargin', 'D_eFGPct', 'D_Pyth',
        'H2H_WinRate', 'EloWinProb', 'SeedWinProb',
        'D_TourneyApps', 'D_DeepRunRate',
    ]
    for f in must_keep:
        if f in feat_cols and f not in selected:
            selected.append(f)

    # Deduplicate while preserving order
    seen = set(); selected = [f for f in selected if not (f in seen or seen.add(f))]
    print(f'   {gender} selected {len(selected)} / {len(feat_cols)} features '
          f'(shadow threshold={threshold})')
    return selected





def train_gender_model(
        train_df_g : pd.DataFrame,
        val_df_g   : pd.DataFrame,
        feat_cols  : list,
        gender     : str,
) -> dict:
    """
    Train a LightGBM + XGBoost ensemble for one gender.

    Returns a dict with keys:
        lgb_model, xgb_model, iso_cal,
        w_lgb, w_xgb,
        lgb_brier, xgb_brier, ens_brier, cal_brier,
        lgb_cv_model, xgb_cv_model,
        val_preds_cal, val_preds_ens
    """
    X_tr  = train_df_g[feat_cols]
    y_tr  = train_df_g['Outcome']
    X_val = val_df_g[feat_cols]
    y_val = val_df_g['Outcome']

    # ── Recency sample weights ────────────────────────────────────
    # Exponential decay: recent seasons count more than old ones.
    # Formula: w_i = decay ^ (max_season - season_i)
    #   2025 → 1.000,  2024 → 0.800,  2023 → 0.640,
    #   2022 → 0.512,  2020 → 0.328,  2015 → 0.107, ...
    #
    # Validation rows are ALWAYS weight=1.0 — the isotonic calibrator
    # fits on validation predictions, so no weighting needed there.
    #
    # The augmented (flipped) rows get the same weight as the original
    # row they came from (same season), so symmetry is preserved.
    decay      = CFG['recency_decay']
    max_season = int(train_df_g['Season'].max())
    w_tr       = decay ** (max_season - train_df_g['Season'].values)
    # Normalise so mean weight = 1.0 (keeps effective learning rate stable)
    w_tr       = w_tr / w_tr.mean()

    # Full dataset: train + val — val rows get weight 1.0
    all_df   = pd.concat([train_df_g, val_df_g], ignore_index=True)
    X_all    = all_df[feat_cols]
    y_all    = all_df['Outcome']
    max_s_all = int(all_df['Season'].max())
    w_all    = decay ** (max_s_all - all_df['Season'].values)
    w_all    = w_all / w_all.mean()

    print(f'\n   {gender}: train={len(X_tr):,}  val={len(X_val):,}  '
          f'(weight range [{w_tr.min():.3f}, {w_tr.max():.3f}])')

    # ── Optuna Bayesian Hyperparameter Tuning ─────────────────
    # Uses temporal leave-one-season-out CV for unbiased estimates.
    # We tune on the training split only (not val) to prevent leakage.
    # Recency weights applied during scoring for consistency.
    # 25/20/15 trials keeps runtime ~8-12 min for the whole pipeline.
    print(f'\n   Tuning {gender} hyperparameters (Optuna TPE) ...')

    X_np  = X_tr.values if hasattr(X_tr, 'values') else np.array(X_tr)
    y_np  = y_tr.values if hasattr(y_tr, 'values') else np.array(y_tr)
    s_np  = train_df_g['Season'].values
    w_np  = w_tr

    # Use a mutable list as shared state between closures —
    # more reliable than `global` inside nested functions.
    _state = [None]   # _state[0] will hold the latest prediction array

    def _temporal_brier(model_fn, X, y, seasons, weights):
        """Leave-one-season-out Brier on training seasons only."""
        unique_s = sorted(np.unique(seasons))
        if len(unique_s) < 4:
            return 0.25  # not enough seasons to CV
        preds, trues = [], []
        for ts in unique_s[-6:]:   # only last 6 seasons for speed
            tr_idx = np.where(seasons < ts)[0]
            te_idx = np.where(seasons == ts)[0]
            if len(tr_idx) < 40 or len(te_idx) == 0:
                continue
            w_fold = weights[tr_idx]
            model_fn(X[tr_idx], y[tr_idx], w_fold, X[te_idx])
            preds.append(_state[0])
            trues.append(y[te_idx])
        if not preds:
            return 0.25
        return brier_score_loss(np.concatenate(trues), np.concatenate(preds))

    # ── LightGBM Optuna ───────────────────────────────────────
    def _lgb_objective(trial):
        p = {
            'objective'        : 'binary',
            'metric'           : 'binary_logloss',
            'verbose'          : -1,
            'random_state'     : CFG['seed'],
            'learning_rate'    : trial.suggest_float('lr',  0.01, 0.10, log=True),
            'num_leaves'       : trial.suggest_int('nl',    8, 64),
            'max_depth'        : trial.suggest_int('md',    3, 7),
            'min_child_samples': trial.suggest_int('mcs',   5, 50),
            'feature_fraction' : trial.suggest_float('ff',  0.5, 1.0),
            'bagging_fraction' : trial.suggest_float('bf',  0.5, 1.0),
            'bagging_freq'     : 5,
            'reg_alpha'        : trial.suggest_float('ra',  1e-3, 5.0, log=True),
            'reg_lambda'       : trial.suggest_float('rl',  1e-3, 10.0, log=True),
        }
        def _fit(Xtr, ytr, wtr, Xte):
            ds = lgb.Dataset(Xtr, label=ytr, weight=wtr)
            m  = lgb.train(p, ds, num_boost_round=300, callbacks=[lgb.log_evaluation(-1)])
            _state[0] = m.predict(Xte)
        return _temporal_brier(_fit, X_np, y_np, s_np, w_np)

    sampler = TPESampler(seed=CFG['seed'], n_startup_trials=8, multivariate=True)
    lgb_study = optuna.create_study(direction='minimize', sampler=sampler)
    lgb_study.optimize(_lgb_objective, n_trials=15, show_progress_bar=False)
    lgb_p_opt = lgb_study.best_params
    lgb_p = {
        'objective'        : 'binary',
        'metric'           : 'binary_logloss',
        'verbose'          : -1,
        'random_state'     : CFG['seed'],
        'bagging_freq'     : 5,
        'learning_rate'    : lgb_p_opt['lr'],
        'num_leaves'       : lgb_p_opt['nl'],
        'max_depth'        : lgb_p_opt['md'],
        'min_child_samples': lgb_p_opt['mcs'],
        'feature_fraction' : lgb_p_opt['ff'],
        'bagging_fraction' : lgb_p_opt['bf'],
        'reg_alpha'        : lgb_p_opt['ra'],
        'reg_lambda'       : lgb_p_opt['rl'],
    }
    print(f'   {gender} LGB best CV Brier={lgb_study.best_value:.5f}')

    # ── XGBoost Optuna ────────────────────────────────────────
    def _xgb_objective(trial):
        p = {
            'objective'      : 'binary:logistic',
            'eval_metric'    : 'logloss',
            'verbosity'      : 0,
            'seed'           : CFG['seed'],
            'learning_rate'  : trial.suggest_float('lr', 0.01, 0.10, log=True),
            'max_depth'      : trial.suggest_int('md',   3, 7),
            'subsample'      : trial.suggest_float('ss', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('cb', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('mcw', 1, 15),
            'gamma'          : trial.suggest_float('g',  0.0, 3.0),
            'reg_alpha'      : trial.suggest_float('ra', 1e-3, 5.0, log=True),
            'reg_lambda'     : trial.suggest_float('rl', 1e-3, 10.0, log=True),
        }
        def _fit(Xtr, ytr, wtr, Xte):
            dm  = xgb.DMatrix(Xtr, label=ytr, weight=wtr)
            m   = xgb.train(p, dm, num_boost_round=300)
            _state[0] = m.predict(xgb.DMatrix(Xte))
        return _temporal_brier(_fit, X_np, y_np, s_np, w_np)

    xgb_study = optuna.create_study(direction='minimize',
                                    sampler=TPESampler(seed=CFG['seed']+1,
                                                       n_startup_trials=7,
                                                       multivariate=True))
    xgb_study.optimize(_xgb_objective, n_trials=12, show_progress_bar=False)
    xgb_p_opt = xgb_study.best_params
    xgb_p = {
        'objective'        : 'binary:logistic',
        'eval_metric'      : 'logloss',
        'verbosity'        : 0,
        'seed'             : CFG['seed'],
        'learning_rate'    : xgb_p_opt['lr'],
        'max_depth'        : xgb_p_opt['md'],
        'subsample'        : xgb_p_opt['ss'],
        'colsample_bytree' : xgb_p_opt['cb'],
        'min_child_weight' : xgb_p_opt['mcw'],
        'gamma'            : xgb_p_opt['g'],
        'reg_alpha'        : xgb_p_opt['ra'],
        'reg_lambda'       : xgb_p_opt['rl'],
    }
    print(f'   {gender} XGB best CV Brier={xgb_study.best_value:.5f}')

    # ── CatBoost Optuna ───────────────────────────────────────
    def _cat_objective(trial):
        p = {
            'verbose'            : 0,
            'random_seed'        : CFG['seed'],
            'iterations'         : 300,
            'learning_rate'      : trial.suggest_float('lr',  0.01, 0.10, log=True),
            'depth'              : trial.suggest_int('d',     3, 7),
            'l2_leaf_reg'        : trial.suggest_float('l2',  1.0, 20.0, log=True),
            'bagging_temperature': trial.suggest_float('bt',  0.0, 2.0),
            'random_strength'    : trial.suggest_float('rs',  0.0, 2.0),
        }
        def _fit(Xtr, ytr, wtr, Xte):
            m = CatBoostClassifier(**p)
            m.fit(Xtr, ytr, sample_weight=wtr, verbose=False)
            _state[0] = m.predict_proba(Xte)[:, 1]
        return _temporal_brier(_fit, X_np, y_np, s_np, w_np)

    cat_study = optuna.create_study(direction='minimize',
                                    sampler=TPESampler(seed=CFG['seed']+2,
                                                       n_startup_trials=6,
                                                       multivariate=True))
    cat_study.optimize(_cat_objective, n_trials=25, show_progress_bar=False)
    cat_p_opt = cat_study.best_params
    cat_p = {
        'verbose'            : 0,
        'random_seed'        : CFG['seed'],
        'iterations'         : 600,   # more iterations for final model
        'learning_rate'      : cat_p_opt['lr'],
        'depth'              : cat_p_opt['d'],
        'l2_leaf_reg'        : cat_p_opt['l2'],
        'bagging_temperature': cat_p_opt['bt'],
        'random_strength'    : cat_p_opt['rs'],
    }
    print(f'   {gender} CAT best CV Brier={cat_study.best_value:.5f}')

    # ── LightGBM final fit ─────────────────────────────────────
    lgb_tr_ds  = lgb.Dataset(X_tr,  label=y_tr,  weight=w_tr)
    lgb_val_ds = lgb.Dataset(X_val, label=y_val)

    lgb_cv = lgb.train(
        lgb_p, lgb_tr_ds,
        num_boost_round = 3000,
        valid_sets      = [lgb_val_ds],
        callbacks       = [
            lgb.early_stopping(75, verbose=False),
            lgb.log_evaluation(200),
        ],
    )
    lgb_val_p = lgb_cv.predict(X_val)
    lgb_b     = brier_score_loss(y_val, lgb_val_p)
    print(f'   {gender} LightGBM  Brier={lgb_b:.5f}  iter={lgb_cv.best_iteration}')

    lgb_full = lgb.train(lgb_p, lgb.Dataset(X_all, label=y_all, weight=w_all),
                         num_boost_round=lgb_cv.best_iteration)

    # ── XGBoost final fit ──────────────────────────────────────
    dtr  = xgb.DMatrix(X_tr,  label=y_tr,  weight=w_tr)
    dvl  = xgb.DMatrix(X_val, label=y_val)
    dall = xgb.DMatrix(X_all, label=y_all, weight=w_all)

    xgb_cv = xgb.train(
        xgb_p, dtr,
        num_boost_round       = 3000,
        evals                 = [(dvl, 'val')],
        early_stopping_rounds = 75,
        
    )
    xgb_val_p = xgb_cv.predict(dvl)
    xgb_b     = brier_score_loss(y_val, xgb_val_p)
    print(f'   {gender} XGBoost   Brier={xgb_b:.5f}  iter={xgb_cv.best_iteration}')

    xgb_full = xgb.train(xgb_p, dall,
                         num_boost_round=xgb_cv.best_iteration)

    # ── CatBoost final fit ─────────────────────────────────────
    cat_cv = CatBoostClassifier(**cat_p)
    cat_cv.fit(
        X_tr, y_tr,
        sample_weight         = w_tr,
        eval_set              = (X_val, y_val),
        early_stopping_rounds = 50,
    )
    cat_val_p = cat_cv.predict_proba(X_val)[:, 1]
    cat_b     = brier_score_loss(y_val, cat_val_p)
    print(f'   {gender} CatBoost  Brier={cat_b:.5f}  iter={cat_cv.best_iteration_}')

    cat_full = CatBoostClassifier(**cat_p)
    cat_full.fit(X_all, y_all, sample_weight=w_all)

    # ── LogisticRegression (4th ensemble member) ───────────────
    # Adds linear decision boundary diversity. Reference shows it
    # contributes ~equal weight to boosted models in the blend.
    # Impute + scale needed since LR is sensitive to scale.
    _imp = SimpleImputer(strategy='median')
    _sca = StandardScaler()
    X_tr_lr  = _sca.fit_transform(_imp.fit_transform(X_tr))
    X_val_lr = _sca.transform(_imp.transform(X_val))
    X_all_lr = _sca.transform(_imp.transform(X_all))

    lr_cv = LogisticRegression(C=0.5, penalty='l2', solver='lbfgs',
                                max_iter=2000, random_state=CFG['seed'])
    lr_cv.fit(X_tr_lr, y_tr, sample_weight=w_tr)
    lr_val_p = lr_cv.predict_proba(X_val_lr)[:, 1]
    lr_b     = brier_score_loss(y_val, lr_val_p)
    print(f'   {gender} LogReg    Brier={lr_b:.5f}')

    lr_full = LogisticRegression(C=0.5, penalty='l2', solver='lbfgs',
                                  max_iter=2000, random_state=CFG['seed'])
    lr_full.fit(X_all_lr, y_all, sample_weight=w_all)

    # ── Ensemble (inverse-Brier weights: LGB + XGB + CAT + LR) ──
    inv_l = 1.0 / lgb_b
    inv_x = 1.0 / xgb_b
    inv_c = 1.0 / cat_b
    inv_r = 1.0 / lr_b
    tot   = inv_l + inv_x + inv_c + inv_r
    wl    = inv_l / tot
    wx    = inv_x / tot
    wc    = inv_c / tot
    wr    = inv_r / tot

    ens_val_p = wl*lgb_val_p + wx*xgb_val_p + wc*cat_val_p + wr*lr_val_p
    ens_b     = brier_score_loss(y_val, ens_val_p)
    print(f'   {gender} Ensemble  Brier={ens_b:.5f}  '
          f'(LGB={wl:.2f} XGB={wx:.2f} CAT={wc:.2f} LR={wr:.2f})')

    # ── Isotonic calibration ────────────────────────────────────
    iso_g = IsotonicRegression(out_of_bounds='clip')
    iso_g.fit(ens_val_p, y_val)
    cal_val_p = iso_g.predict(ens_val_p)
    cal_b     = brier_score_loss(y_val, cal_val_p)
    print(f'   {gender} Calibrated Brier={cal_b:.5f}  (Δ={cal_b-ens_b:+.5f})')

    return {
        'lgb_model'    : lgb_full,
        'xgb_model'    : xgb_full,
        'cat_model'    : cat_full,
        'lr_model'     : lr_full,
        'lr_imp'       : _imp,
        'lr_sca'       : _sca,
        'iso_cal'      : iso_g,
        'w_lgb'        : wl,
        'w_xgb'        : wx,
        'w_cat'        : wc,
        'w_lr'         : wr,
        'lgb_brier'    : lgb_b,
        'xgb_brier'    : xgb_b,
        'cat_brier'    : cat_b,
        'lr_brier'     : lr_b,
        'ens_brier'    : ens_b,
        'cal_brier'    : cal_b,
        'lgb_cv_model' : lgb_cv,
        'xgb_cv_model' : xgb_cv,
        'val_preds_ens': ens_val_p,
        'val_preds_cal': cal_val_p,
        'y_val'        : y_val.values,
        'X_val'        : X_val,
        'feat_cols'    : feat_cols,
    }


# ── Split training data by gender ────────────────────────────────
val_mask = train_df['Season'].isin(CFG['val_seasons'])

m_train_cut = train_df[(~val_mask) & (train_df['IsWomen'] == 0)].copy()
m_val_cut   = train_df[( val_mask) & (train_df['IsWomen'] == 0)].copy()
w_train_cut = train_df[(~val_mask) & (train_df['IsWomen'] == 1)].copy()
w_val_cut   = train_df[( val_mask) & (train_df['IsWomen'] == 1)].copy()

# Full datasets (train + val) used for final model retraining
m_all_df    = train_df[train_df['IsWomen'] == 0].copy()
w_all_df    = train_df[train_df['IsWomen'] == 1].copy()

print(f'   Men  : train={len(m_train_cut):,}  val={len(m_val_cut):,}  '
      f'total={len(m_all_df):,}')
print(f'   Women: train={len(w_train_cut):,}  val={len(w_val_cut):,}  '
      f'total={len(w_all_df):,}')

# ── Shadow feature selection (now that splits exist) ─────────────
M_FEAT_COLS = shadow_feature_selection(
    m_all_df, FEAT_COLS, "Men's",   CFG['val_seasons'], threshold=0.60)
W_FEAT_COLS = shadow_feature_selection(
    w_all_df, FEAT_COLS, "Women's", CFG['val_seasons'], threshold=0.60)

# ── Train both models ─────────────────────────────────────────────
# Wrapped in try/except: if ANYTHING crashes during training, we fall
# back to a simple Elo-only submission so Kaggle always gets a CSV.
_TRAIN_OK = False
try:
    m_models = train_gender_model(m_train_cut, m_val_cut, M_FEAT_COLS, "Men's")
    w_models = train_gender_model(w_train_cut, w_val_cut, W_FEAT_COLS, "Women's")
    _TRAIN_OK = True
except Exception as _e:
    print(f'\n⚠️  Training crashed: {_e}')
    print('   Falling back to Elo-only submission ...')
    import traceback; traceback.print_exc()

if not _TRAIN_OK:
    # ── Elo fallback: guaranteed output ───────────────────────────
    # Build a minimal Elo-based submission so Kaggle accepts the run.
    def _elo_prob(t1, t2, season, elo_lu, init=CFG['elo_init']):
        e1 = elo_lu.get((season, t1), init)
        e2 = elo_lu.get((season, t2), init)
        return float(np.clip(1.0 / (1.0 + 10**((e2-e1)/400)), 0.025, 0.975))

    for sub_path, sub_df_src in [
        (OUTPUT / 'submission_stage1_v15.csv', s1_matchups),
        (OUTPUT / 'submission_stage2_v15.csv', s2_matchups),
    ]:
        rows = []
        for _, r in sub_df_src.iterrows():
            parts = str(r['ID']).split('_')
            season, t1, t2 = int(parts[0]), int(parts[1]), int(parts[2])
            is_w = season >= 2010 and t1 > 3000
            lu = w_elo_lu if is_w else m_elo_lu
            rows.append({'ID': r['ID'],
                         'Pred': _elo_prob(t1, t2, season, lu)})
        pd.DataFrame(rows).to_csv(sub_path, index=False)
        print(f'   ✅ Fallback saved: {sub_path.name}  ({len(rows):,} rows)')
    raise SystemExit(0)   # exit cleanly so Kaggle shows the output

# ── Combined validation summary ──────────────────────────────────
# Merge val predictions from both genders to compute overall Brier
combined_val_preds = np.concatenate([
    m_models['val_preds_cal'], w_models['val_preds_cal']])
combined_val_y     = np.concatenate([
    m_models['y_val'], w_models['y_val']])
overall_cal_brier  = brier_score_loss(combined_val_y, combined_val_preds)

print(f'\n   ══════════════════════════════════════════')
print(f"   Men's   calibrated Brier : {m_models['cal_brier']:.5f}")
print(f"   Women's calibrated Brier : {w_models['cal_brier']:.5f}")
print(f'   Overall calibrated Brier : {overall_cal_brier:.5f}')
print(f'   ══════════════════════════════════════════')

# ── Convenience aliases used by diagnostics & submission ─────────
# These let downstream code reference a single set of names while
# still dispatching to the correct gender model at prediction time.
ens_val_preds = np.concatenate([
    m_models['val_preds_ens'], w_models['val_preds_ens']])
cal_val_preds = combined_val_preds
y_VAL         = combined_val_y
ens_brier     = brier_score_loss(y_VAL, ens_val_preds)
cal_brier     = overall_cal_brier

# Kept for diagnostics feature importance plots
lgb_cv_model  = m_models['lgb_cv_model']   # use men's as representative
xgb_cv_model  = m_models['xgb_cv_model']
ALL_FEAT_COLS = M_FEAT_COLS   # same columns for both genders


# ================================================================
# SECTION 9 — Prediction & Submission
# ================================================================
print(f'\n[9/9] Generating submission ...')


def predict_batch(
        matchups_df : pd.DataFrame,
        models_dict : dict,
        clip_lo     : float = CFG['clip_lo'],
        clip_hi     : float = CFG['clip_hi'],
) -> np.ndarray:
    """
    Predict win probabilities for a batch of matchups.

    Dispatches to the correct gender model based on the IsWomen column.
    Men's and Women's rows are predicted separately, then recombined
    in the original row order to preserve alignment with the ID column.

    Parameters
    ----------
    matchups_df : DataFrame with columns in FEAT_COLS + ['IsWomen']
    models_dict : {'men': m_models, 'women': w_models}
    """
    preds = np.full(len(matchups_df), np.nan)

    for gender_flag, key in [(0, 'men'), (1, 'women')]:
        mask = (matchups_df['IsWomen'].values == gender_flag)
        if mask.sum() == 0:
            continue
        mdl       = models_dict[key]
        feat_cols = mdl['feat_cols']   # each gender has its own selected cols
        sub       = matchups_df.loc[mask, feat_cols]
        p_lgb = mdl['lgb_model'].predict(sub)
        p_xgb = mdl['xgb_model'].predict(xgb.DMatrix(sub))
        p_cat = mdl['cat_model'].predict_proba(sub)[:, 1]
        # LogReg needs imputation + scaling (fitted on training data)
        sub_lr = mdl['lr_sca'].transform(mdl['lr_imp'].transform(sub))
        p_lr   = mdl['lr_model'].predict_proba(sub_lr)[:, 1]
        ens    = (mdl['w_lgb'] * p_lgb
                + mdl['w_xgb'] * p_xgb
                + mdl['w_cat'] * p_cat
                + mdl['w_lr']  * p_lr)
        cal   = mdl['iso_cal'].predict(ens)
        preds[mask] = cal

    return np.clip(preds, clip_lo, clip_hi)


# Build the models dict used by predict_batch
MODELS = {'men': m_models, 'women': w_models}


def build_features_for_submission(sub_df: pd.DataFrame,
                                   label: str) -> pd.DataFrame:
    """
    Build feature rows for every matchup in a submission DataFrame.
    Processes in chunks of 5,000 to keep peak memory flat.

    Parameters
    ----------
    sub_df : DataFrame with columns [ID, Season, T1, T2]
    label  : display label for the tqdm progress bar

    Returns
    -------
    DataFrame with all feature columns + ID column, ready for predict_batch.
    """
    CHUNK  = 5_000
    chunks = []
    n      = len(sub_df)

    for start in tqdm(range(0, n, CHUNK), desc=f'   {label}'):
        chunk = sub_df.iloc[start:start + CHUNK]
        chunk_rows = []
        for _, r in chunk.iterrows():
            s, t1, t2 = r['Season'], int(r['T1']), int(r['T2'])
            is_women  = (t1 >= 3000)
            feats = build_matchup_features(s, t1, t2, is_women)
            feats['Season']  = s
            feats['IsWomen'] = int(is_women)
            feats['ID']      = r['ID']
            chunk_rows.append(feats)
        chunks.append(pd.DataFrame(chunk_rows))
        del chunk_rows

    result = pd.concat(chunks, ignore_index=True)
    del chunks
    gc.collect()
    return result


# ── Generate predictions (wrapped for safety) ─────────────────────
s1_path   = OUTPUT / 'submission_stage1_v15.csv'
s2_path   = OUTPUT / 'submission_stage2_v15.csv'
comb_path = OUTPUT / 'submission_combined_v15.csv'

try:
    print(f'   Building Stage-1 features ({len(sub1):,} matchups) ...')
    s1_feat_df  = build_features_for_submission(sub1, 'Stage-1')
    s1_preds    = predict_batch(s1_feat_df, MODELS)
    s1_sub      = pd.DataFrame({'ID': s1_feat_df['ID'], 'Pred': s1_preds})
    s1_is_women = s1_feat_df['IsWomen'].values.copy()
    del s1_feat_df; gc.collect()

    print(f'   Building Stage-2 features ({len(sub2):,} matchups) ...')
    s2_feat_df  = build_features_for_submission(sub2, 'Stage-2')
    s2_preds    = predict_batch(s2_feat_df, MODELS)
    s2_sub      = pd.DataFrame({'ID': s2_feat_df['ID'], 'Pred': s2_preds})
    s2_is_women = s2_feat_df['IsWomen'].values.copy()
    del s2_feat_df; gc.collect()

    combined = pd.concat([s1_sub, s2_sub], ignore_index=True)
    s1_sub.to_csv(s1_path,   index=False)
    s2_sub.to_csv(s2_path,   index=False)
    combined.to_csv(comb_path, index=False)

    preds      = s1_preds
    submission = s1_sub
    sub_path   = s1_path

    print(f'\n   ✅ Submissions saved:')
    print(f'      {s1_path.name}  — {len(s1_sub):,} rows  <- SUBMIT THIS NOW')
    print(f'      {s2_path.name}  — {len(s2_sub):,} rows  <- submit after Selection Sunday')
    print(f'      {comb_path.name} — {len(combined):,} rows  <- full combined')
    print(f'   Stage-1 Pred mean  : {s1_preds.mean():.4f}  (should be ~0.50)')
    print(f'   Stage-1 Pred std   : {s1_preds.std():.4f}')
    print(f'   Stage-1 Pred range : [{s1_preds.min():.4f}, {s1_preds.max():.4f}]')
    print(f'   Stage-2 Pred mean  : {s2_preds.mean():.4f}')
    print(f'   Clipped at         : [{CFG["clip_lo"]}, {CFG["clip_hi"]}]')

except Exception as _pred_e:
    print(f'\n⚠️  Prediction crashed: {_pred_e}')
    import traceback; traceback.print_exc()
    print('   Writing Elo-only fallback submissions ...')

    def _elo_prob_fb(t1, t2, season):
        lu = w_elo_lu if t1 > 3000 else m_elo_lu
        e1 = lu.get((season, t1), CFG['elo_init'])
        e2 = lu.get((season, t2), CFG['elo_init'])
        return float(np.clip(1.0/(1.0+10**((e2-e1)/400)), 0.025, 0.975))

    for path, src in [(s1_path, sub1), (s2_path, sub2)]:
        rows = []
        for _, r in src.iterrows():
            p = str(r['ID']).split('_')
            rows.append({'ID': r['ID'],
                         'Pred': _elo_prob_fb(int(p[1]), int(p[2]), int(p[0]))})
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f'   ✅ Fallback: {path.name}  ({len(rows):,} rows)')

    pd.concat([pd.read_csv(s1_path), pd.read_csv(s2_path)]).to_csv(comb_path, index=False)
    s1_sub = pd.read_csv(s1_path)
    s2_sub = pd.read_csv(s2_path)
    preds  = s1_sub['Pred'].values
    submission = s1_sub
    sub_path   = s1_path


# ================================================================
# SECTION 10 — Diagnostics & Visualisations
# ================================================================
print('\n[Diag] Generating diagnostic plots ...')

fig = plt.figure(figsize=(22, 16),
                 facecolor='#0f172a')
fig.suptitle('March Mania 2026  —  Model Diagnostics',
             fontsize=16, fontweight='bold', color='#f8fafc', y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig,
                        hspace=0.45, wspace=0.35)

C = ['#38bdf8','#f472b6','#34d399','#fb923c','#a78bfa','#fbbf24']
ax_kw = dict(facecolor='#1e293b')

# ── 1. Calibration curve ─────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0], **ax_kw)
frac, mean_pred = calibration_curve(y_VAL, ens_val_preds, n_bins=10)
frac_c, mean_c  = calibration_curve(y_VAL, cal_val_preds, n_bins=10)
ax.plot(mean_pred, frac,   'o--', color=C[0], lw=1.5, ms=5,
        label='Uncalibrated')
ax.plot(mean_c,    frac_c, 's-',  color=C[1], lw=2,   ms=6,
        label='Calibrated')
ax.plot([0,1],[0,1], '--', color='#64748b', lw=1, label='Perfect')
ax.set_title('Calibration Curve', color='#f8fafc')
ax.set_xlabel('Mean Predicted Prob', color='#cbd5e1')
ax.set_ylabel('Fraction of Positives', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=8)

# ── 2. Prediction distribution ───────────────────────────────────
ax = fig.add_subplot(gs[0, 1], **ax_kw)
_s1p = s1_sub['Pred'].values if 's1_preds' not in dir() else s1_preds
_s1w = np.zeros(len(_s1p)) if 's1_is_women' not in dir() else s1_is_women
ax.hist(_s1p[_s1w == 0], bins=50,
        color=C[0], alpha=0.7, density=True, label="Men's")
ax.hist(_s1p[_s1w == 1], bins=50,
        color=C[1], alpha=0.7, density=True, label="Women's")
ax.set_title('Prediction Distribution (Stage-1)', color='#f8fafc')
ax.set_xlabel('Predicted Probability', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=8)

# ── 3. Feature importance — LightGBM ─────────────────────────────
ax = fig.add_subplot(gs[0, 2], **ax_kw)
fi_lgb = pd.Series(
    lgb_cv_model.feature_importance(importance_type='gain'),
    index=ALL_FEAT_COLS
).nlargest(20).sort_values()
ax.barh(range(len(fi_lgb)), fi_lgb.values,
        color=C[2], edgecolor='#475569')
ax.set_yticks(range(len(fi_lgb)))
ax.set_yticklabels(fi_lgb.index, fontsize=7.5)
ax.set_title('LightGBM Feature Importance (Gain, Top 20)',
             color='#f8fafc')
ax.tick_params(colors='#94a3b8')

# ── 4. Feature importance — XGBoost ──────────────────────────────
ax = fig.add_subplot(gs[1, 0], **ax_kw)
fi_xgb = pd.Series(
    xgb_cv_model.get_score(importance_type='gain')
).nlargest(20).sort_values()
ax.barh(range(len(fi_xgb)), fi_xgb.values,
        color=C[3], edgecolor='#475569')
ax.set_yticks(range(len(fi_xgb)))
ax.set_yticklabels(fi_xgb.index, fontsize=7.5)
ax.set_title('XGBoost Feature Importance (Gain, Top 20)',
             color='#f8fafc')
ax.tick_params(colors='#94a3b8')

# ── 5. Brier score by season (val) ───────────────────────────────
# Rebuild val_cut2 from the gender-specific val DataFrames that are
# still in scope (m_val_cut, w_val_cut), combined with their
# calibrated predictions stored in the models dicts.
ax = fig.add_subplot(gs[1, 1], **ax_kw)
val_cut2 = pd.concat([
    m_val_cut[['Season','Outcome']].assign(
        pred    = m_models['val_preds_cal'],
        IsWomen = 0),
    w_val_cut[['Season','Outcome']].assign(
        pred    = w_models['val_preds_cal'],
        IsWomen = 1),
], ignore_index=True)
val_cut2['sq_err'] = (val_cut2['pred'] - val_cut2['Outcome']) ** 2
bs_by_season = val_cut2.groupby('Season')['sq_err'].mean()
ax.bar(bs_by_season.index, bs_by_season.values,
       color=C[4], edgecolor='#475569', width=0.6)
ax.axhline(0.25, color='#fbbf24', ls='--', lw=1.5,
           label='Random (0.25)')
ax.set_title('Brier Score by Validation Season',
             color='#f8fafc')
ax.set_xlabel('Season', color='#cbd5e1')
ax.set_ylabel('Brier Score', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=8)
for x, v in zip(bs_by_season.index, bs_by_season.values):
    ax.text(x, v + 0.002, f'{v:.4f}', ha='center',
            fontsize=8, color='#cbd5e1')

# ── 6. Gender-level Brier ─────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2], **ax_kw)
for label, flag, color in [("Men's", 0, C[0]), ("Women's", 1, C[1])]:
    mask = val_cut2['IsWomen'] == flag
    bs_g = val_cut2[mask].groupby('Season')['sq_err'].mean()
    ax.plot(bs_g.index, bs_g.values, marker='o', lw=2,
            ms=6, color=color, label=label)
ax.axhline(0.25, color='#fbbf24', ls='--', lw=1.5)
ax.set_title("Brier Score by Gender & Season",
             color='#f8fafc')
ax.set_xlabel('Season', color='#cbd5e1')
ax.set_ylabel('Brier Score', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.legend(fontsize=9)

# ── 7. Top 20 men's teams by 2026 Elo ────────────────────────────
ax = fig.add_subplot(gs[2, 0:2], **ax_kw)
m_2026 = (m_elo_df[m_elo_df['Season'] == m_elo_df['Season'].max()]
            .merge(m_teams[['TeamID','TeamName']], on='TeamID')
            .nlargest(20, 'EloPreTourney')
            .sort_values('EloPreTourney', ascending=True))
ax.barh(range(len(m_2026)), m_2026['EloPreTourney'],
        color=C[0], edgecolor='#475569', alpha=0.85)
ax.set_yticks(range(len(m_2026)))
ax.set_yticklabels(m_2026['TeamName'], fontsize=8.5)
ax.set_title("Top 20 Men's Teams — 2026 Pre-Tourney Elo",
             color='#f8fafc')
ax.set_xlabel('Elo Rating', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.axvline(CFG['elo_init'], color='#fbbf24', ls='--',
           lw=1.5, label='Baseline 1500')
ax.legend(fontsize=8)

# ── 8. Top 20 women's teams by 2026 Elo ─────────────────────────
ax = fig.add_subplot(gs[2, 2], **ax_kw)
w_2026 = (w_elo_df[w_elo_df['Season'] == w_elo_df['Season'].max()]
            .merge(w_teams[['TeamID','TeamName']], on='TeamID')
            .nlargest(20, 'EloPreTourney')
            .sort_values('EloPreTourney', ascending=True))
ax.barh(range(len(w_2026)), w_2026['EloPreTourney'],
        color=C[1], edgecolor='#475569', alpha=0.85)
ax.set_yticks(range(len(w_2026)))
ax.set_yticklabels(w_2026['TeamName'], fontsize=8.5)
ax.set_title("Top 20 Women's Teams — 2026 Pre-Tourney Elo",
             color='#f8fafc')
ax.set_xlabel('Elo Rating', color='#cbd5e1')
ax.tick_params(colors='#94a3b8')
ax.axvline(CFG['elo_init'], color='#fbbf24', ls='--', lw=1.5)

for a in fig.axes:
    a.set_facecolor('#1e293b')
    for spine in a.spines.values():
        spine.set_edgecolor('#334155')
    a.grid(color='#334155', linestyle='--', alpha=0.4)

plt.savefig(OUTPUT / 'diagnostics_v15.png', dpi=150,
            bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.close()

# ── Print top-20 rankings ─────────────────────────────────────────
print(f'\n  Top 20 Men\'s (Elo {m_elo_df["Season"].max()}):')
print(m_2026[::-1][['TeamName','EloPreTourney']].to_string(index=False))

print(f'\n  Top 20 Women\'s (Elo {w_elo_df["Season"].max()}):')
print(w_2026[::-1][['TeamName','EloPreTourney']].to_string(index=False))

# ── Final summary ─────────────────────────────────────────────────
elapsed = time.time() - t0
print(f'\n{DIVIDER}')
print(f'  PIPELINE COMPLETE   {elapsed:.1f}s')
print(f'{DIVIDER}')
print(f'  Men\'s  LGB Brier    : {m_models["lgb_brier"]:.5f}')
print(f'  Men\'s  XGB Brier    : {m_models["xgb_brier"]:.5f}')
print(f'  Men\'s  CAT Brier    : {m_models["cat_brier"]:.5f}')
print(f'  Men\'s  LR  Brier    : {m_models["lr_brier"]:.5f}')
print(f'  Men\'s  Ens Brier    : {m_models["ens_brier"]:.5f}')
print(f'  Men\'s  Cal Brier    : {m_models["cal_brier"]:.5f}')
print(f'  Men\'s  Weights      : '
      f'LGB={m_models["w_lgb"]:.3f} XGB={m_models["w_xgb"]:.3f} '
      f'CAT={m_models["w_cat"]:.3f} LR={m_models["w_lr"]:.3f}')
print(f'  Women\'s LGB Brier   : {w_models["lgb_brier"]:.5f}')
print(f'  Women\'s XGB Brier   : {w_models["xgb_brier"]:.5f}')
print(f'  Women\'s CAT Brier   : {w_models["cat_brier"]:.5f}')
print(f'  Women\'s LR  Brier   : {w_models["lr_brier"]:.5f}')
print(f'  Women\'s Ens Brier   : {w_models["ens_brier"]:.5f}')
print(f'  Women\'s Cal Brier   : {w_models["cal_brier"]:.5f}')
print(f'  Women\'s Weights     : '
      f'LGB={w_models["w_lgb"]:.3f} XGB={w_models["w_xgb"]:.3f} '
      f'CAT={w_models["w_cat"]:.3f} LR={w_models["w_lr"]:.3f}')
print(f'  Overall Cal Brier   : {overall_cal_brier:.5f}')
print(f'  Stage-1 rows        : {len(s1_sub):,}')
print(f'  Stage-2 rows        : {len(s2_sub):,}')
print(f'  Outputs → {OUTPUT}')
print(f'    • submission_stage1_v15.csv  <- SUBMIT THIS')
print(f'    • submission_stage2_v15.csv  <- after Selection Sunday')
print(f'    • diagnostics_v15.png')
print(f'{DIVIDER}')
print()
print('  ── Next Steps ──────────────────────────────────────────')
print('  1. Submit submission_stage1_v15.csv on Kaggle')
print('  2. Share LB score here → we iterate further')
print(f'{DIVIDER}')

──────────────────────────────────────────────────────────────────────
  March Machine Learning Mania 2026  —  Pipeline v2.4
──────────────────────────────────────────────────────────────────────

[1/9] Loading data ...
   Men  regular season : 196,823 games (compact) + detailed loaded lazily
   Women regular season: 140,825 games (compact) + detailed loaded lazily
   Men  tournament      :   2,585 games
   Women tournament     :   1,717 games
   Massey entries       : 5,761,702
   Stage-1 matchups     : 519,144  <- active submission window
   Stage-2 matchups     : 132,133  <- scored after tourney starts

[2/9] Computing Elo ratings ...
   Men  Elo table: 14,206 team-seasons
   Women Elo table: 9,952 team-seasons

[3/9] Building Massey Ordinals features ...
   Massey: 8,355 team-seasons | systems: ['POM', 'SAG', 'COL', 'DOL', 'MOR', 'WLK', 'RTH']

[4/9] Computing Four Factors & box-score features ...
   Loading Men's detailed (lazy, usecols only) ...
   Four Factors: 8,346 team-season

  Women tourney rows: 100%|██████████| 1087/1087 [00:00<00:00, 11297.45it/s]



   Base training rows   : 2,216
   After augmentation   : 4,432 (2× — T1/T2 symmetric)
   Men   (2008–2025): 1,129 tournament games
   Women (2008–2025): 1,087 tournament games
   Class balance        : 0.500 (should be exactly 0.50 after augmentation)

[8/9] Training models (separate Men's & Women's) ...
   Men  : train=1,856  val=402  total=2,258
   Women: train=1,772  val=402  total=2,174
   Men's feature selection on 1,188 games (9 early seasons) ...
   Men's selected 18 / 99 features (shadow threshold=0.6)
   Women's feature selection on 1,134 games (9 early seasons) ...
   Women's selected 18 / 99 features (shadow threshold=0.6)

   Men's: train=1,856  val=402  (weight range [0.005, 6.504])

   Tuning Men's hyperparameters (Optuna TPE) ...
   Men's LGB best CV Brier=0.13142
   Men's XGB best CV Brier=0.16626
   Men's CAT best CV Brier=0.04979
[200]	valid_0's binary_logloss: 0.461202
[400]	valid_0's binary_logloss: 0.35973
   Men's LightGBM  Brier=0.10905  iter=511
[0]	val-loglos

   Stage-1: 100%|██████████| 104/104 [00:52<00:00,  2.00it/s]


   Building Stage-2 features (132,133 matchups) ...


   Stage-2: 100%|██████████| 27/27 [00:12<00:00,  2.12it/s]



   ✅ Submissions saved:
      submission_stage1_v15.csv  — 519,144 rows  <- SUBMIT THIS NOW
      submission_stage2_v15.csv  — 132,133 rows  <- submit after Selection Sunday
      submission_combined_v15.csv — 651,277 rows  <- full combined
   Stage-1 Pred mean  : 0.5062  (should be ~0.50)
   Stage-1 Pred std   : 0.3616
   Stage-1 Pred range : [0.0250, 0.9750]
   Stage-2 Pred mean  : 0.5331
   Clipped at         : [0.025, 0.975]

[Diag] Generating diagnostic plots ...

  Top 20 Men's (Elo 2026):
    TeamName  EloPreTourney
        Duke    1790.816487
     Houston    1783.678655
 Connecticut    1756.169993
     Arizona    1752.909249
     Gonzaga    1740.155678
     Florida    1735.282828
      Purdue    1718.783952
 Michigan St    1714.351212
     Iowa St    1709.599024
    Michigan    1704.135861
    Illinois    1702.480080
St Mary's CA    1699.958500
   St John's    1699.759774
      Auburn    1695.307928
   Tennessee    1694.241198
     Alabama    1686.866844
     Clemson    1685.6